# E5 Dataset-Induced Biomedical Simplification Graph

CLEF SimpleText Task 1.1 biomedical text simplification.

This notebook automatically induces a lightweight simplification knowledge graph from the training set. It does not manually encode biomedical concepts, definitions, or graph edges.

## Objective

Discover observed term simplification patterns by comparing each `complex` sentence with its paired `simple` sentence.

Example:

`Patients with hypertension received anticoagulants.`

`Patients with high blood pressure received blood thinners.`

Candidate relations:

- `hypertension -> high blood pressure`
- `anticoagulants -> blood thinners`

The graph nodes are discovered terms, and each edge uses the relation `simplified_as`.

## Approach

The notebook uses alignment heuristics rather than a manually curated ontology:

1. Load all training sentence pairs.
2. Tokenize and normalize each complex/simple pair.
3. Use sequence alignment to find replacement spans.
4. Keep candidate substitutions where the complex span looks more technical than the simple span.
5. Aggregate frequencies and supporting sentence pairs.
6. Remove general wording changes such as `there was -> we found`.
7. Keep strict term-level edges with minimum frequency >= 1.

This graph is an empirical project artifact: it emerges from observed transformations in the dataset.

## Imports and Paths

In [ ]:
from __future__ import annotations

import pickle
import re
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_PATH = PROJECT_ROOT / "data" / "sentence_no_context" / "train_clean.csv"
KG_DIR = PROJECT_ROOT / "data" / "knowledge_graph"
FIGURES_DIR = PROJECT_ROOT / "figures"

NODES_PATH = KG_DIR / "nodes.csv"
EDGES_PATH = KG_DIR / "edges.csv"
GPICKLE_PATH = KG_DIR / "biomedical_knowledge_graph.gpickle"
GRAPHML_PATH = KG_DIR / "biomedical_knowledge_graph.graphml"
TOP_RELATIONS_FIGURE_PATH = FIGURES_DIR / "knowledge_graph_top_relations.png"

KG_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MIN_EDGE_FREQUENCY = 1
MAX_SPAN_TOKENS = 6

print(f"Project root: {PROJECT_ROOT}")
print(f"Training file: {TRAIN_PATH.relative_to(PROJECT_ROOT)}")
print(f"Knowledge graph directory: {KG_DIR.relative_to(PROJECT_ROOT)}")

## Load Training Data

Only `complex` and `simple` are used. No labels, context, document fields, or manual terms are used.

In [ ]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Missing training data: {TRAIN_PATH}")

train_df = pd.read_csv(TRAIN_PATH)
required_columns = ["complex", "simple"]
missing_columns = [column for column in required_columns if column not in train_df.columns]
if missing_columns:
    raise ValueError(f"train_clean.csv is missing columns: {missing_columns}")

pairs_df = train_df[required_columns].copy()
pairs_df["complex"] = pairs_df["complex"].fillna("").astype(str).str.strip()
pairs_df["simple"] = pairs_df["simple"].fillna("").astype(str).str.strip()
pairs_df = pairs_df[pairs_df["complex"].ne("") & pairs_df["simple"].ne("")].reset_index(drop=True)
pairs_df["pair_index"] = pairs_df.index

print(f"Loaded training pairs: {len(pairs_df):,}")
display(pairs_df.head())

## Text Normalization and Tokenization

In [ ]:
TOKEN_PATTERN = re.compile(r"[A-Za-z]+(?:[-'][A-Za-z]+)?|\d+(?:\.\d+)?%?")

STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "of", "in", "on", "to", "for", "with", "without",
    "by", "from", "at", "as", "than", "then", "that", "this", "these", "those", "was", "were",
    "is", "are", "be", "been", "being", "had", "has", "have", "having", "did", "does", "do",
    "not", "no", "we", "our", "they", "their", "patients", "patient", "participants", "participant",
    "people", "person", "group", "groups", "study", "studies", "trial", "trials", "review",
    "evidence", "result", "results", "outcome", "outcomes",
}

GENERAL_WORDS = STOPWORDS | {
    "more", "less", "few", "fewer", "many", "most", "some", "any", "all", "may", "might", "can",
    "could", "would", "should", "showed", "found", "reported", "included", "compared", "received",
    "reduced", "increased", "improved", "different", "difference", "important", "small", "large",
    "required", "needed", "suggest", "suggested", "conclude", "concluded", "available", "current",
}

NON_TERM_SOURCE_TOKENS = GENERAL_WORDS | {
    "there", "here", "also", "however", "therefore", "thus", "overall", "probably", "possibly",
    "brief", "long", "short", "term", "follow-up", "using", "used", "use", "care", "usual",
    "identified", "identify", "assessed", "assess", "comparing", "compare", "versu", "versus",
    "further", "including", "include", "unable", "able", "significant", "supplemented", "supplement",
    "required", "require", "available", "observed", "evaluated", "evaluate", "provided", "provide",
    "receiving", "receive", "conducted", "conduct", "determine", "determined", "involving", "involved",
    "regarding", "remaining", "although", "appeared", "appear", "approximately", "event",
}

NON_TERM_TARGET_TOKENS = STOPWORDS | {
    "found", "compared", "compared to", "more", "such as", "not able", "clear", "needed",
    "containing", "included", "including", "reported", "showed", "seen", "planned",
    "who received", "carried out", "show", "involved", "about", "other", "however", "seemed",
    "there", "use", "place", "consider", "curtain", "controlled", "greater", "important",
}

DEMOGRAPHIC_OR_STUDY_POPULATION_TERMS = {
    "participant", "participants", "patient", "patients", "person", "people", "adult", "adults",
    "child", "children", "woman", "women", "man", "men", "mother", "mothers", "infant", "infants",
}

NON_BIOMEDICAL_SOURCE_TERMS = DEMOGRAPHIC_OR_STUDY_POPULATION_TERMS | {
    "comparison", "effectiveness", "clinical", "analogue", "approximately",
}

NON_TERM_SOURCE_PHRASES = {
    "there was", "there were", "we found", "we included", "we did", "we are", "it wa", "it is",
    "this review", "this study", "our review", "the review", "the study",
}


def normalize_text(text: str) -> str:
    text = text.lower()
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str) -> list[str]:
    return [token.lower() for token in TOKEN_PATTERN.findall(normalize_text(text))]


def singularize_token(token: str) -> str:
    if len(token) > 4 and token.endswith("ies"):
        return token[:-3] + "y"
    if len(token) > 3 and token.endswith("s") and not token.endswith("ss"):
        return token[:-1]
    return token


def normalize_phrase(tokens: list[str]) -> str:
    normalized = [singularize_token(token.lower()) for token in tokens]
    return " ".join(normalized).strip()


def phrase_word_count(phrase: str) -> int:
    return len(phrase.split())

print(tokenize("Patients with hypertension received anticoagulants."))
print(normalize_phrase(tokenize("anticoagulants")))

## Alignment Heuristics

Replacement spans are extracted with `difflib.SequenceMatcher`. A candidate is kept only when the complex-side phrase looks like a biomedical, statistical, or clinical term. This prevents generic rewrites such as `administered -> gave` or `there was -> we found` from entering the graph.

In [ ]:
def technical_score(phrase: str) -> float:
    tokens = phrase.split()
    if not tokens:
        return 0.0
    score = 0.0
    for token in tokens:
        if token in GENERAL_WORDS:
            score -= 0.5
        if len(token) >= 8:
            score += 1.0
        if any(part in token for part in ["itis", "osis", "emia", "pathy", "card", "renal", "pulmon", "thromb", "hyper", "hypo", "anti", "glyc", "myocard", "mortality", "morbidity"]):
            score += 1.5
        if "-" in token:
            score += 0.3
    if len(tokens) == 1:
        score += 0.5
    return score


def simple_score(phrase: str) -> float:
    tokens = phrase.split()
    if not tokens:
        return 0.0
    score = 0.0
    for token in tokens:
        if token in GENERAL_WORDS:
            score += 0.4
        if len(token) <= 6:
            score += 0.3
    if len(tokens) > 1:
        score += 0.7
    return score



BIOMEDICAL_MARKERS = {
    "mortality", "morbidity", "adverse", "event", "incidence", "prevalence", "pulmonary",
    "corticosteroid", "analgesia", "cerclage", "arteriosu", "ductu", "lmwh", "bmmnc",
    "diabetic", "neuropathy", "radiotherapy", "antibiotic", "clinical", "renal", "cardiac",
    "myocardial", "hypothyroidism", "hyperthyroidism", "fracture", "steroid", "cancer",
    "exacerbation", "glucose", "antibody", "nephrectomy", "neurodevelopmental", "progestin",
    "progesterone", "prophylaxi", "rhtsh", "transfusion", "thyrotropin", "ductus",
}

STATISTICAL_TERM_MARKERS = {
    "proportion", "rate", "ratio", "risk", "efficacy", "effectiveness", "sample size",
    "certainty", "low-certainty", "high-certainty", "outcome", "response", "comparison",
}

DOMAIN_TERM_EXACT = {
    "intervention", "proportion", "incidence", "prevalence", "mortality", "morbidity",
    "adverse event", "sample size", "clinical response", "patent ductu arteriosu",
}


def has_biomedical_marker(tokens: list[str]) -> bool:
    joined = " ".join(tokens)
    return any(marker in joined for marker in BIOMEDICAL_MARKERS)


def marker_matches_phrase(markers: set[str], phrase: str, tokens: list[str], allow_substring: bool = False) -> bool:
    for marker in markers:
        marker_tokens = marker.split()
        if len(marker_tokens) > 1 and marker in phrase:
            return True
        if len(marker_tokens) == 1:
            if marker in tokens:
                return True
            if allow_substring and any(marker in token for token in tokens):
                return True
    return False


def domain_term_score(phrase: str) -> float:
    tokens = phrase.split()
    joined = " ".join(tokens)
    score = 0.0
    if joined in DOMAIN_TERM_EXACT:
        score += 3.0
    if marker_matches_phrase(BIOMEDICAL_MARKERS, joined, tokens, allow_substring=True):
        score += 2.0
    if marker_matches_phrase(STATISTICAL_TERM_MARKERS, joined, tokens, allow_substring=False):
        score += 1.5
    # Acronyms are lowercased by tokenization, so catch compact technical abbreviations.
    if len(tokens) == 1 and len(tokens[0]) <= 6 and re.search(r"[a-z]*[bcdfghjklmnpqrstvwxyz]{3,}[a-z]*", tokens[0]):
        score += 1.0
    if any(len(token) >= 10 for token in tokens):
        score += 0.8
    if any(token in NON_TERM_SOURCE_TOKENS for token in tokens):
        score -= 1.0
    if any(token in STOPWORDS for token in tokens) and joined not in DOMAIN_TERM_EXACT:
        score -= 2.0
    return score


def looks_like_term_source(phrase: str) -> bool:
    tokens = phrase.split()
    if not tokens:
        return False
    if phrase in NON_TERM_SOURCE_PHRASES:
        return False
    if phrase in NON_BIOMEDICAL_SOURCE_TERMS:
        return False
    if tokens[0] in {"there", "we", "it", "this", "that", "these", "those", "the", "a", "an"}:
        return False
    if all(token in NON_TERM_SOURCE_TOKENS for token in tokens):
        return False
    if len(tokens) == 1 and tokens[0] in NON_TERM_SOURCE_TOKENS:
        return False
    if any(token in DEMOGRAPHIC_OR_STUDY_POPULATION_TERMS for token in tokens) and not has_biomedical_marker(tokens):
        return False
    if domain_term_score(phrase) < 1.5:
        return False
    return True


def looks_like_simplification_target(phrase: str) -> bool:
    tokens = phrase.split()
    if not tokens:
        return False
    if phrase in NON_TERM_SOURCE_PHRASES:
        return False
    if phrase in NON_BIOMEDICAL_SOURCE_TERMS:
        return False
    if phrase in NON_TERM_TARGET_TOKENS:
        return False
    if any(token in STOPWORDS for token in tokens) and phrase not in DOMAIN_TERM_EXACT:
        return False
    if all(token in NON_TERM_TARGET_TOKENS for token in tokens):
        return False
    if len(tokens) == 1 and len(tokens[0]) < 4:
        return False
    return True

def valid_candidate(complex_phrase: str, simple_phrase: str) -> bool:
    if not complex_phrase or not simple_phrase:
        return False
    if complex_phrase == simple_phrase:
        return False
    if phrase_word_count(complex_phrase) > MAX_SPAN_TOKENS or phrase_word_count(simple_phrase) > MAX_SPAN_TOKENS:
        return False
    if len(complex_phrase) < 4 or len(simple_phrase) < 3:
        return False
    if complex_phrase in GENERAL_WORDS or simple_phrase in STOPWORDS:
        return False
    if not looks_like_term_source(complex_phrase):
        return False
    if not looks_like_simplification_target(simple_phrase):
        return False
    if set(complex_phrase.split()).issubset(set(simple_phrase.split())):
        return False
    if set(simple_phrase.split()).issubset(set(complex_phrase.split())) and phrase_word_count(simple_phrase) == 1:
        return False

    if domain_term_score(complex_phrase) < 1.5:
        return False

    complex_technical = technical_score(complex_phrase)
    simple_readable = simple_score(simple_phrase)
    length_pattern = len(complex_phrase) >= len(simple_phrase) or phrase_word_count(simple_phrase) > phrase_word_count(complex_phrase)
    return (complex_technical + simple_readable >= 1.2) and length_pattern


def extract_replacements(complex_sentence: str, simple_sentence: str) -> list[tuple[str, str]]:
    complex_tokens = tokenize(complex_sentence)
    simple_tokens = tokenize(simple_sentence)
    matcher = SequenceMatcher(a=complex_tokens, b=simple_tokens, autojunk=False)
    candidates = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        complex_phrase = normalize_phrase(complex_tokens[i1:i2])
        simple_phrase = normalize_phrase(simple_tokens[j1:j2])
        if valid_candidate(complex_phrase, simple_phrase):
            candidates.append((complex_phrase, simple_phrase))

    return candidates

examples = [
    ("Patients with hypertension received anticoagulants.", "Patients with high blood pressure received blood thinners."),
    ("Mortality was reduced.", "Death was reduced."),
]
for complex_sentence, simple_sentence in examples:
    print(complex_sentence)
    print(simple_sentence)
    print(extract_replacements(complex_sentence, simple_sentence))

## Process Entire Training Set

In [ ]:
edge_support: dict[tuple[str, str], dict[str, Any]] = defaultdict(lambda: {"frequency": 0, "supporting_pair_ids": [], "examples": []})

for row in pairs_df.itertuples(index=False):
    replacements = extract_replacements(row.complex, row.simple)
    seen_in_pair = set()
    for complex_phrase, simple_phrase in replacements:
        key = (complex_phrase, simple_phrase)
        edge_support[key]["frequency"] += 1
        if key not in seen_in_pair:
            edge_support[key]["supporting_pair_ids"].append(int(row.pair_index))
            seen_in_pair.add(key)
        if len(edge_support[key]["examples"]) < 3:
            edge_support[key]["examples"].append({"complex": row.complex, "simple": row.simple})

raw_edges = []
for (complex_phrase, simple_phrase), info in edge_support.items():
    raw_edges.append(
        {
            "source": complex_phrase,
            "relation": "simplified_as",
            "target": simple_phrase,
            "frequency": info["frequency"],
            "supporting_sentence_pairs": len(set(info["supporting_pair_ids"])),
            "supporting_pair_ids": ";".join(map(str, sorted(set(info["supporting_pair_ids"])))) ,
            "example_complex": info["examples"][0]["complex"] if info["examples"] else "",
            "example_simple": info["examples"][0]["simple"] if info["examples"] else "",
        }
    )

raw_edges_df = pd.DataFrame(raw_edges).sort_values(
    ["frequency", "supporting_sentence_pairs", "source"],
    ascending=[False, False, True],
).reset_index(drop=True)

print(f"Raw candidate edges: {len(raw_edges_df):,}")
display(raw_edges_df.head(20))

## Filter Noisy Edges

Strict term-like substitutions are retained even when observed once. General sentence rewrites, demographic/study-population substitutions such as `participant -> woman`, and non-term phrases such as `there was`, `we found`, `identified`, `comparing`, standalone stopwords, and generic target rewrites are filtered out before aggregation.

In [ ]:
edges_df = raw_edges_df[raw_edges_df["frequency"] >= MIN_EDGE_FREQUENCY].copy().reset_index(drop=True)

if len(edges_df) == 0:
    raise ValueError("No edges survived filtering. Lower MIN_EDGE_FREQUENCY or inspect extraction heuristics.")

node_terms = sorted(set(edges_df["source"]) | set(edges_df["target"]))
nodes_df = pd.DataFrame(
    [
        {
            "node_id": re.sub(r"[^a-z0-9]+", "_", term).strip("_"),
            "term": term,
            "node_type": "discovered_term",
            "source_frequency": int(edges_df.loc[edges_df["source"].eq(term), "frequency"].sum()),
            "target_frequency": int(edges_df.loc[edges_df["target"].eq(term), "frequency"].sum()),
        }
        for term in node_terms
    ]
)

# Keep graph ids stable even when two terms normalize to the same id.
id_counts = Counter()
stable_ids = []
for node_id in nodes_df["node_id"]:
    id_counts[node_id] += 1
    stable_ids.append(node_id if id_counts[node_id] == 1 else f"{node_id}_{id_counts[node_id]}")
nodes_df["node_id"] = stable_ids
term_to_node_id = dict(zip(nodes_df["term"], nodes_df["node_id"], strict=True))

edges_df["source_id"] = edges_df["source"].map(term_to_node_id)
edges_df["target_id"] = edges_df["target"].map(term_to_node_id)

print(f"Filtered edges: {len(edges_df):,}")
print(f"Discovered nodes: {len(nodes_df):,}")
display(edges_df.head(50)[["source", "target", "frequency", "supporting_sentence_pairs", "example_complex", "example_simple"]])

## Top 50 Discovered Simplification Relations

In [ ]:
top_50_relations_df = edges_df.head(50)[["source", "target", "frequency", "supporting_sentence_pairs"]].copy()
top_50_relations_df["relation"] = top_50_relations_df["source"] + " -> " + top_50_relations_df["target"]
display(top_50_relations_df[["relation", "frequency", "supporting_sentence_pairs"]])

plt.figure(figsize=(10, 12))
plot_df = top_50_relations_df.sort_values("frequency", ascending=True)
plt.barh(plot_df["relation"], plot_df["frequency"])
plt.xlabel("Frequency")
plt.title("Top discovered simplification relations")
plt.tight_layout()
plt.savefig(TOP_RELATIONS_FIGURE_PATH, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved figure: {TOP_RELATIONS_FIGURE_PATH.relative_to(PROJECT_ROOT)}")

## Build Graph

In [ ]:
def validate_induced_graph_tables(nodes: pd.DataFrame, edges: pd.DataFrame) -> None:
    if nodes["node_id"].duplicated().any():
        duplicates = nodes.loc[nodes["node_id"].duplicated(), "node_id"].tolist()
        raise ValueError(f"Duplicate node ids: {duplicates}")
    if edges.duplicated(subset=["source_id", "relation", "target_id"]).any():
        raise ValueError("Duplicate graph edges found after filtering.")
    node_ids = set(nodes["node_id"])
    missing_sources = sorted(set(edges["source_id"]) - node_ids)
    missing_targets = sorted(set(edges["target_id"]) - node_ids)
    if missing_sources:
        raise ValueError(f"Missing source nodes: {missing_sources}")
    if missing_targets:
        raise ValueError(f"Missing target nodes: {missing_targets}")

validate_induced_graph_tables(nodes_df, edges_df)

G = nx.DiGraph()
for row in nodes_df.itertuples(index=False):
    G.add_node(
        row.node_id,
        term=row.term,
        node_type=row.node_type,
        source_frequency=int(row.source_frequency),
        target_frequency=int(row.target_frequency),
    )

for row in edges_df.itertuples(index=False):
    G.add_edge(
        row.source_id,
        row.target_id,
        relation="simplified_as",
        frequency=int(row.frequency),
        supporting_sentence_pairs=int(row.supporting_sentence_pairs),
    )

print(f"Number of nodes: {G.number_of_nodes():,}")
print(f"Number of edges: {G.number_of_edges():,}")
print(f"Average degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")

degree_df = pd.DataFrame(
    [{"term": G.nodes[node]["term"], "degree": degree} for node, degree in G.degree()]
).sort_values("degree", ascending=False).reset_index(drop=True)
print("Top connected nodes:")
display(degree_df.head(20))

## Save Graph Files

In [ ]:
nodes_df.to_csv(NODES_PATH, index=False)
edges_df.to_csv(EDGES_PATH, index=False)

with GPICKLE_PATH.open("wb") as file:
    pickle.dump(G, file)

nx.write_graphml(G, GRAPHML_PATH)

print(f"Saved nodes: {NODES_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved edges: {EDGES_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved graph pickle: {GPICKLE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved graphml: {GRAPHML_PATH.relative_to(PROJECT_ROOT)}")

## Helper Functions

The helper functions query only the filtered induced graph. If an illustrative medical pair such as `hypertension -> high blood pressure` is not observed at least twice in `train_clean.csv`, it is not returned. This keeps the graph dataset-induced rather than manually corrected.

In [ ]:
def get_term_simplification(term: str, top_k: int = 1) -> str | list[str] | None:
    normalized_term = normalize_phrase(tokenize(term))
    matches = edges_df[edges_df["source"].eq(normalized_term)].sort_values(
        ["frequency", "supporting_sentence_pairs"],
        ascending=[False, False],
    )
    if len(matches) == 0:
        return None
    simplifications = matches["target"].head(top_k).tolist()
    return simplifications[0] if top_k == 1 else simplifications


def phrase_variants(phrase: str) -> set[str]:
    normalized = normalize_phrase(tokenize(phrase))
    variants = {normalized}
    tokens = normalized.split()
    if len(tokens) == 1:
        token = tokens[0]
        variants.add(token + "s")
        if token.endswith("y"):
            variants.add(token[:-1] + "ies")
    return {variant for variant in variants if variant}


def get_sentence_simplifications(sentence: str) -> list[str]:
    normalized_sentence = " ".join(tokenize(sentence))
    found = []
    seen = set()
    for row in edges_df.sort_values("frequency", ascending=False).itertuples(index=False):
        variants = phrase_variants(row.source)
        if any(re.search(rf"(?<![a-z0-9]){re.escape(variant)}(?![a-z0-9])", normalized_sentence) for variant in variants):
            statement = f"{row.source} -> {row.target}"
            if statement not in seen:
                found.append(statement)
                seen.add(statement)
    return found

print("mortality:", get_term_simplification("mortality"))
print("adverse event:", get_term_simplification("adverse events"))
print(get_sentence_simplifications("Mortality and adverse events were reported."))

## Inspect Example Sentences

In [ ]:
example_sentences = [
    "Mortality and adverse events were reported.",
    "Mortality was reduced.",
    "The confidence interval was wide.",
    "Participants underwent angioplasty after myocardial infarction.",
    "Glucocorticoids reduced airway inflammation.",
]

example_outputs_df = pd.DataFrame(
    [
        {
            "sentence": sentence,
            "discovered_simplifications": get_sentence_simplifications(sentence),
        }
        for sentence in example_sentences
    ]
)
display(example_outputs_df)

## Limitations

This graph is induced automatically from observed complex-to-simple sentence pairs. It is not a manually curated medical ontology and it is not guaranteed to be medically complete or always correct.

The extraction method uses simple alignment heuristics, so it can miss valid paraphrases and may retain noisy substitutions. The `frequency >= 2` filter reduces noise but does not remove all errors.

This notebook does not train BioBART and does not evaluate simplification metrics. It only discovers term-level simplification knowledge from the dataset and saves it for later knowledge-enhanced generation experiments.